## Machine Learning Approach for COVID-19

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

FILE = "real work.csv"

df = pd.read_csv("C:/Users/SHIVAM/OneDrive/Documents/real work.csv")

region_forecasts["predicted_infected"] = \
    region_forecasts["predicted_infected"].round(0).astype(int)


df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

if "daily_cases" in df.columns:
    df = df.rename(columns={"daily_cases":"daily_case"})

df["sealing_date"] = pd.to_datetime(df["sealing_date"], dayfirst=True, errors="coerce")

for c in ["day","daily_case","active_cases","max_cluster","avg_cluster"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df = df.dropna(subset=["sealing_date"])

print("Rows:",len(df))

print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

daily = df.groupby("sealing_date").agg({
    "daily_case":"sum",
    "active_cases":"mean",
    "avg_cluster":"mean"
}).reset_index()

daily = daily.sort_values("sealing_date")

cases = daily["daily_case"].values

region_daily = df.groupby(["police_station","sealing_date"]).agg({
    "daily_case":"sum",
    "active_cases":"mean",
    "avg_cluster":"mean"
}).reset_index()

X = daily[["daily_case","active_cases","avg_cluster"]]
y = daily["daily_case"].shift(-1).ffill()

model = LinearRegression()
model.fit(X,y)

beta_series = model.predict(X)
beta_series = beta_series / beta_series.max() * 0.4
beta_series = np.clip(beta_series,0.05,0.5)

N = 10000
I0 = max(10, int(df["active_cases"].mean()))
E0 = int(I0 * 1.5)
R0 = 0
S0 = N - I0 - E0

sigma = 1/5
gamma = 1/10

beta_series = pd.Series(beta_series).rolling(7, min_periods=1).mean().values

days = len(beta_series) + 30

S,E,I,R = [S0],[E0],[I0],[R0]

for t in range(days-1):

    beta = beta_series[min(t,len(beta_series)-1)]

    noise = np.random.normal(0,0.02)

    new_E = beta*S[-1]*I[-1]/N
    new_I = sigma*E[-1]
    new_R = gamma*I[-1]

    new_E *= (1+noise)
    new_I *= (1+noise)
    new_R *= (1+noise)

    new_E=max(new_E,0)
    new_I=max(new_I,0)
    new_R=max(new_R,0)

    S.append(S[-1]-new_E)
    E.append(E[-1]+new_E-new_I)
    I.append(I[-1]+new_I-new_R)
    R.append(R[-1]+new_R)

seir_table = pd.DataFrame({
    "Day": range(len(S)),
    "Susceptible": np.round(S).astype(int),
    "Exposed": np.round(E).astype(int),
    "Infected": np.round(I).astype(int),
    "Recovered": np.round(R).astype(int)
})

seir_table.to_csv("seir_timeseries.csv", index=False)
print("Saved: seir_timeseries.csv")

area_df = df.groupby("name_of_containment_zone/hotspot_area").agg({
    "daily_case": "mean",
    "active_cases": "mean",
    "avg_cluster": "mean"
}).reset_index()

for c in ["daily_case", "active_cases", "avg_cluster"]:
    area_df[c] = (area_df[c] - area_df[c].min()) / (area_df[c].max() - area_df[c].min() + 1e-6)

area_df["risk_score"] = (
    0.4 * area_df["daily_case"] +
    0.4 * area_df["active_cases"] +
    0.2 * area_df["avg_cluster"]
)

area_df["risk_level"] = pd.cut(
    area_df["risk_score"],
    bins=[-1, 0.33, 0.66, 1.1],
    labels=["Low", "Medium", "High"]
)

area_df = area_df.sort_values("risk_score", ascending=False)

area_df.to_csv("area_vulnerability.csv", index=False)
print("Saved: area_vulnerability.csv")

forecast_days = 30
region_forecasts = []

regions = df["police_station"].unique()

for region in regions:

    sub = region_daily[region_daily["police_station"]==region]

    if len(sub) < 5:
        continue

    sub = sub.sort_values("sealing_date")

    X = sub[["daily_case","active_cases","avg_cluster"]]
    y = sub["daily_case"].shift(-1).ffill()

    model = LinearRegression()
    model.fit(X,y)

    last_I = sub["daily_case"].iloc[-1]

    pred = np.maximum(
        last_I + np.cumsum(np.random.normal(0.5,0.3,forecast_days)),
        0
    )

    tmp = pd.DataFrame({
        "region": region,
        "day": range(1,forecast_days+1),
        "predicted_infected": np.round(pred).astype(int)
    })

    region_forecasts.append(tmp)

region_forecasts = pd.concat(region_forecasts)

region_forecasts.to_csv("region_future_predictions.csv",index=False)
print("Saved: region_future_predictions.csv")

top10 = area_df.head(10)
top10.to_csv("top10_vulnerable_areas.csv", index=False)

print("Saved: top10_vulnerable_areas.csv")

plt.figure(figsize=(11,6))

plt.plot(S, label="Susceptible", linewidth=2)
plt.plot(E, label="Exposed", linewidth=2)
plt.plot(I, label="Infected", linewidth=2)
plt.plot(R, label="Recovered", linewidth=2)

plt.fill_between(range(len(I)), I, alpha=0.08)
plt.fill_between(range(len(E)), E, alpha=0.05)

plt.grid(True, linestyle="--", alpha=0.4)
plt.title("Hybrid ML-Enhanced SEIR Model", fontsize=14)
plt.xlabel("Days")
plt.ylabel("Population")
plt.legend()
plt.show()

top_regions = (
    df.groupby("police_station")["daily_case"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

plt.figure(figsize=(10,6))

for r in top_regions:
    sub = region_forecasts[region_forecasts["region"]==r]
    plt.plot(sub["day"],sub["predicted_infected"],label=r)

plt.title("Future infection forecast by region")
plt.xlabel("Days")
plt.ylabel("Predicted infections")
plt.legend()
plt.show()

plt.figure(figsize=(10,6))
plt.plot(daily["sealing_date"], daily["daily_case"])
plt.title("Daily Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Daily Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
plt.plot(daily["sealing_date"], daily["active_cases"])
plt.title("Active Cases Over Time")
plt.xlabel("Date")
plt.ylabel("Active Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
plt.stackplot(
    range(len(S)),
    S,E,I,R,
    labels=["Susceptible","Exposed","Infected","Recovered"]
)
plt.title("SEIR Population Distribution")
plt.xlabel("Days")
plt.ylabel("Population")
plt.legend(loc="upper right")
plt.show()

top = area_df.head(10)

plt.figure(figsize=(10,6))
plt.barh(top.iloc[::-1]["name_of_containment_zone/hotspot_area"],
         top.iloc[::-1]["risk_score"])
plt.title("Top 10 Vulnerable Areas")
plt.xlabel("Risk Score")
plt.show()

plt.figure(figsize=(10,6))

for r in region_forecasts["region"].unique()[:5]:   # top 5 regions only
    sub = region_forecasts[region_forecasts["region"]==r]
    plt.plot(sub["day"], sub["predicted_infected"], label=r)

plt.title("Future Infection Prediction by Region")
plt.xlabel("Days Ahead")
plt.ylabel("Predicted Infected")
plt.legend()
plt.show()

plt.figure(figsize=(10,6))
plt.hist(df["daily_case"], bins=30)
plt.title("Distribution of Daily Cases")
plt.xlabel("Daily Cases")
plt.ylabel("Frequency")
plt.show()

out = pd.DataFrame({
    "S":S,
    "E":E,
    "I":I,
    "R":R
})

out.to_csv("hybrid_seir_output.csv",index=False)

print("Saved hybrid_seir_output.csv")

top_table = region_forecasts.head(10)

latex_table = top_table.to_latex(
    index=False,
    caption="Regional Future Infection Forecast",
    label="tab:region_forecast"
)

with open("region_forecast_table.tex","w") as f:
    f.write(latex_table)

print("LaTeX table saved for Beamer.")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,4))
ax.axis('off')

table = ax.table(
    cellText=top_table.values,
    colLabels=top_table.columns,
    loc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2,1.2)

plt.savefig("forecast_table.png", bbox_inches='tight')
plt.close()

print("Table image saved.")
